In [ ]:
import os
import pandas as pd
import csv
from nltk.tokenize import sent_tokenize
import tweepy

In [ ]:
consumer_key = '..Consumer Key from Twitter API..'
consumer_secret = '..Consumer Secret from Twitter API..'
access_token = '..Access Token from Twitter API..'
access_token_secret = '..Access Token Secret from Twitter API..'

# access to the Twitter API
auth = tweepy.OAuthHandler(consumer_key, consumer_secret)
auth.set_access_token(access_token, access_token_secret)

api = tweepy.API(auth)

# first query keywords
query1 = ["Elgam3a","Elketab","Elcode","Hcode","Codo","ElLab",
        "Elengineering","Betalk","elhaya","el3elm","el3alm",
        "elreada","elweb","becode","ana","hwa","elpower","elkalam",
        "elassignment","mesh","gamed","bas","awee","sana","barra","guys","اللاب",
        "بيكود","shokrann","7ad","Alhamdu lelah","done","elemta7anat","elshoghl","elmoratab",
        "kol","7agaa"]

# second query keywords
query2 = ["by7sl","enharda","yaft7","awel","zone","lya","btsa3dny","hrun",
        "hcalak","hshare","bncode","bnrun","bnstudt","study","nzaker","bnplay",
        "desken","booken","laptopen","grade","akel"]

# fetch first query tweets
with open('query1result.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["#","ID","Created at","Text","language","Research Word"])
    n = 1
    for k in range(0,len(query1)):
        # count for the tweets count and geocode for the location of the tweets which is Cairo
        search_results = api.search(q=query1[k], count=20000, geocode="30.04442,31.235712,100km")

        print ('get all data for ',query1[k])

        tmp=[]  
        tweets_for_csv = [[tweet.id_str, tweet.created_at,tweet.text,tweet.lang] for tweet in search_results] # CSV file created  
        
        for j in tweets_for_csv: 
            tmp.append(j)  
            
        for i in range(0,len(tmp)):
            print(n,tmp[i][0],tmp[i][1],tmp[i][2],tmp[i][3],query1[k]) 
            writer.writerow([n,tmp[i][0],tmp[i][1],tmp[i][2],tmp[i][3],query1[k]])
            n+=1
            
# fetch second query tweets
with open('query2result.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["#","ID","Created at","Text","language","Research Word"])
    n = 1
    for k in range(0,len(query2)):
        # count for the tweets count and geocode for the location of the tweets which is Cairo
        search_results = api.search(q=query2[k], count=20000, geocode="30.04442,31.235712,100km")

        print ('get all data for ',query2[k])

        tmp=[]  
        tweets_for_csv = [[tweet.id_str, tweet.created_at,tweet.text,tweet.lang] for tweet in search_results] # CSV file created  
        
        for j in tweets_for_csv: 
            tmp.append(j)  
            
        for i in range(0,len(tmp)):
            print(n,tmp[i][0],tmp[i][1],tmp[i][2],tmp[i][3],query2[k]) 
            writer.writerow([n,tmp[i][0],tmp[i][1],tmp[i][2],tmp[i][3],query2[k]])
            n+=1

In [ ]:
# then concatenate the two queries results into one csv file called "twitter.csv"
records = pd.read_csv('twitter.csv')

# remove the duplicates tweets
deduped = records.drop_duplicates(['Text'])

# save the results
deduped.to_csv('twitterRemoveDuplicates.csv', index=False)


In [ ]:
# we decide manually which tweets we will consider by adding another column to the 
# csv file called "State" its values are "accept" if we will take the tweet and "reject" if we will not take it

# also, we added an extra column to the table called lang
# its values are:
# Franco -> if the tweets in English but they were written in Arabic or vise verse.
# switch -> if the tweets have two languages or more.
# arabic -> if the tweets in Arabic and were written in Arabic also.
# english -> if the tweets in English and were written in English also.

records = pd.read_csv('twitterRemoveDuplicates.csv')

# remove the rejected tweets
deduped = records[records['State']=='accept']

# save the results
deduped.to_csv('twitterAfterManualSelect.csv', index=False)


records = pd.read_csv('twitterAfterManualSelect.csv')

# remove the user's names in tweets
def removeUsers(text):
    out = ''
    i=0
    while  (i < len(text)):
        if(text[i]=="@"):
            while (text[i] != " "):
                i+=1
        else:
            out += text[i]
        i+=1
    return out

length = records["Text"].count()

for i in range(0,length):
    records["Text"][i] = removeUsers(records["Text"][i])

# save the results
records.to_csv('twitterAfterRemoveUsers.csv', index=False)

# remove the duplicates tweets
records = pd.read_csv('twitterAfterRemoveUsers.csv')

deduped = records.drop_duplicates(['Text'])

deduped.to_csv('twitterAfterRemoveUsers.csv', index=False)


### Remove the duplicates to faster the process then manually annotate the tokens

In [ ]:
# Tokenized the tweets
records = pd.read_csv('twitterAfterRemoveUsers.csv')

texts = records["Text"]

DataText = open("DataText.txt","w") 

for i in range(0,len(texts)):
    DataText.writelines(texts[i]+"\n")

DataText.close() 

texts = open("DataText.txt","r")
out = texts.read()
out = out.lower()
tokens = out.split()
print(len(tokens))

# save the results to csv file
with open('Tokens.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["#","Token"])
    k = 1
    for i in range(0,len(tokens)):
        # to remove URL
        if(("https://" in tokens[i])==False):
            writer.writerow([k,tokens[i]])
            k = k+1
            print(k)
            
# remove the duplicate to annotate then assign the value to all data to make the process faster.
records = pd.read_csv('Tokens.csv')

deduped = records.drop_duplicates(['Token'])

deduped.to_csv('UniqueTokens.csv', index=False)

# We manually tag and segment the tokens in the file "UniqueTokens.csv"

records = pd.read_csv('UniqueTokens.csv')

# then add the length of the segments
with open('AddingSegmentsLen.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['#','Token','Tag','Tag-Mixed','Segment','SegmentLen'])
    for i in range(0,len(records)):
        if(records['Tag'][i] == 'MIXED' ):
            inp = str(records['Segment'][i]).split(",")
            out = ''
            for j in range(0,len(inp)):
                if (j==0):
                    out += str(len(inp[j]))
                else :
                    out += "," + str(len(inp[j]))
            print(records["#"][i])
            writer.writerow([records['#'][i],records['Token'][i],records['Tag'][i],records['Tag-Mixed'][i],records['Segment'][i],out])
        else:
            out = len(records['Segment'][i])
            writer.writerow([records['#'][i],records['Token'][i],records['Tag'][i],records['Tag-Mixed'][i],records['Segment'][i],out])


# then assign the tags for the in Mixedtag column 
records = pd.read_csv('AddingSegmentsLen.csv')

with open('addingMissingTagsINMixingColumn.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['#','Token','Tag','Tag-Mixed','Segment','SegmentLen'])
    for i in range(0,len(records)):
        if(records['Tag'][i] != 'MIXED' ):
            writer.writerow([records['#'][i],records['Token'][i],records['Tag'][i],records['Tag'][i],records['Segment'][i],records['SegmentLen'][i]])
        else:
            writer.writerow([records['#'][i],records['Token'][i],records['Tag'][i],records['Tag-Mixed'][i],records['Segment'][i],records['SegmentLen'][i]])

# we saved the final tokens in file called "FinalData.csv" 

In [15]:
# load the actual tweets before the tokenization
Twitter= [ os.getcwd() + '/twitterAfterRemoveUsers.csv']
data = pd.read_csv(Twitter[0])

In [16]:
# show an example
data["Text"][7]

'Darwish enta m3ana fe elgam3a wala batlt t3leem khlas?'

In [4]:
import re
pattern = r'<(!?).*>'

In [5]:
# tokenize the tweets
def tokenize(data):
    out = []
    for tweet in data:
        # clean the tokens from URL and spaces and unwanted chars
        tweet = tweet.lower()
        tweet = re.sub(r'^https?:\/\/.*[\r\n]*', '', tweet, flags=re.MULTILINE)
        tweet = re.sub(pattern,'',tweet)
        tweet = re.sub(r'http\S+', '',  tweet)
        l = re.split(r'(\W+)', tweet)
        l[:] = (value for value in l if (value != " " and value != ""))
        out.append(l)
    return out
tokens = tokenize(data["Text"])

In [6]:
# save the tokens with its tweet number
with open('Tokeninsentence.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["#","Sentence #","Token"])
    num = 1
    sentenceNum = 1
    for sentence in tokens:
        for token in sentence:
            writer.writerow([num,sentenceNum,token])
            num += 1
        sentenceNum += 1
    file.close()

In [7]:
# load the final file for unique tokens annotations
TagedToken= [ os.getcwd() + '/FinalData.csv']
data = pd.read_csv(TagedToken[0])
data

,#,Token,Tag,Tag-Mixed,Segment,SegmentLen
0,1,derasa,AR,AR,derasa,6
1,2,?,OTHER,OTHER,?,1
2,3,ya,AR,AR,ya,2
3,4,addy,NE.AR,NE.AR,addy,4
4,5,howa,AR,AR,howa,4
...,...,...,...,...,...,...
3840,3841,confused,EN,EN,confused,8
3841,3842,hahaha,AR,AR,hahaha,6
3842,3843,mafrood,AR,AR,mafrood,7
3843,3844,tekoon,AR,AR,tekoon,6


In [8]:
# load the main tokens file
sentenceToken= [ os.getcwd() + '/Tokeninsentence.csv']
sentData = pd.read_csv(sentenceToken[0])
sentData

,#,Sentence #,Token
0,1,1,derasa
1,2,1,?\n
2,3,1,ya
3,4,1,addy
4,5,1,howa
...,...,...,...
8913,8914,545,el
8914,8915,545,mafrood
8915,8916,545,tekoon
8916,8917,545,fein


In [16]:
# add the annotation and segmentation from unique tokens to the main file
uniquetokens = []
for s in data["Token"]:
    uniquetokens.append(s)
i = 0
with open('TwitterData.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["#","Sentence#","Token","Tag","Tag-Mixed","Segment","SegmentLen"])
    for j in range(len(sentData["Token"])):
        if(sentData["Token"][j] in uniquetokens):
            ind = uniquetokens.index(sentData["Token"][j])
            writer.writerow([sentData["#"][j],sentData["Sentence #"][j],sentData["Token"][j],data["Tag"][ind],data["Tag-Mixed"][ind],data["Segment"][ind],data["SegmentLen"][ind]])
            i += 1
        else:
            writer.writerow([sentData["#"][j],sentData["Sentence #"][j],sentData["Token"][j]])
    file.close()


In [17]:
uniquetokens.index("😺✏")

354

In [18]:
data["Tag"][354]

'OTHER'

In [38]:
# show the output
sentenceToken= [ os.getcwd() + '/TwitterData.csv']
data = pd.read_csv(sentenceToken[0])
data

,#,Sentence#,Token,Tag,Tag-Mixed,Segment,SegmentLen
0,1,1,derasa,AR,AR,derasa,6
1,2,1,?\n,OTHER,OTHER,?\n,NaN
2,3,1,ya,AR,AR,ya,2
3,4,1,addy,NE.AR,NE.AR,addy,4
4,5,1,howa,AR,AR,howa,4
...,...,...,...,...,...,...,...
8650,8651,545,el,AR,AR,el,2
8651,8652,545,mafrood,AR,AR,mafrood,7
8652,8653,545,tekoon,AR,AR,tekoon,6
8653,8654,545,fein,AR,AR,fein,4


In [40]:
# show the mixed words
out = data[data["Tag"]=="MIXED"]
words = list(set(out["Token"].values))
words

['الplastic',
 'عالrevit',
 'blockat',
 'eltweets',
 'والتاجز',
 'eltheme',
 'الrequest',
 'belbeach',
 'الهوت',
 'elfans',
 'elcomfort',
 'elaccount',
 'الموبايل',
 'beecompetition',
 'btsubtweet',
 'afake',
 'betfake',
 'elmatch',
 'elchemistry',
 'elquote',
 'beythrow',
 'الفون',
 'باللاب',
 'الترم',
 'elgoal',
 'أcontrol',
 'b-act',
 'watext',
 'الماوس',
 'اللنتيكال',
 'welmission',
 'elvision',
 'وباص',
 'elendorphins',
 'elserotonin',
 'ال-over',
 'بالموبايل',
 'الاكواد',
 'elsemester',
 'elmaths',
 'ellunch',
 'flconversation',
 'felmatch',
 'elvideo',
 'السوبر',
 'elpost',
 'elacc',
 'بالاب',
 'btargue',
 'elreports',
 'الcase',
 'التاسكات',
 'elterm',
 'الريسرش',
 'elpresentation',
 'والمايك',
 'الداتا',
 'callونتفرج',
 'الfatigability',
 'teplay',
 'elemojis',
 'التليفون',
 'elmixtape',
 'الموبايلات',
 'elmix',
 'اللاب',
 'elgym',
 'acheck',
 'belstars',
 'التيم',
 'للموبايل',
 'elseason',
 'الفايل',
 'متفيرس',
 'بstudy',
 'tchange',
 'الراوتر',
 'الend',
 'الelastic',
 'leon

In [4]:
# show the final twitter data after annotating the tokens without tags
sentenceToken= [ os.getcwd() + '/TwitterData.csv']
data = pd.read_csv(sentenceToken[0])
data

,#,Sentence#,Token,Tag,Tag-Mixed,Segment,SegmentLen
0,1,1,derasa,AR,AR,derasa,6
1,2,1,?\n,OTHER,OTHER,?\n,NaN
2,3,1,ya,AR,AR,ya,2
3,4,1,addy,NE.AR,NE.AR,addy,4
4,5,1,howa,AR,AR,howa,4
...,...,...,...,...,...,...,...
8632,8633,545,el,AR,AR,el,2
8633,8634,545,mafrood,AR,AR,mafrood,7
8634,8635,545,tekoon,AR,AR,tekoon,6
8635,8636,545,fein,AR,AR,fein,4


In [5]:
# some data analysis
print(data.isnull().any())
print(data["Tag"].value_counts())
data["Sentence#"][0]
data['Segment'][0]

#             False
Sentence#     False
Token         False
Tag           False
Tag-Mixed      True
Segment       False
SegmentLen     True
dtype: bool
AR       5206
EN       2321
OTHER     796
MIXED     167
NE.AR      80
NE.EN      48
AMBIG      18
LANG3       1
Name: Tag, dtype: int64


'derasa'

In [10]:
# update the segments length column
mixedLen = 0
with open('TwitterData.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['#',"Sentence#",'Token','Tag','Tag-Mixed','Segment','SegmentLen'])
    for i in range(0,len(data["Token"])):
        if(data['Tag'][i] == 'MIXED' ):
            mixedLen+=1
            inp = str(data['Segment'][i]).split(",")
            out = ''
            for j in range(0,len(inp)):
                if (j==0):
                    out += str(len(inp[j]))
                else :
                    out += "," + str(len(inp[j]))
            writer.writerow([data['#'][i],data['Sentence#'][i],data['Token'][i],data['Tag'][i],data['Tag-Mixed'][i],data['Segment'][i],out])
        else:
            out = len(data['Segment'][i])
            writer.writerow([data['#'][i],data['Sentence#'][i],data['Token'][i],data['Tag'][i],data['Tag'][i],data['Segment'][i],out])
mixedLen

167

In [1]:
# remove the spaces and the enters and unwanted char in the segments
def removespaceIntoken(outputName,inputName):
    Data = pd.read_csv(os.getcwd() + '/'+ inputName +'.csv')
    with open(outputName+'.csv', 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['#', 'Sentence#', 'Token', 'Tag', 'Tag-Mixed', 'Segment', 'SegmentLen'])
        j = 1
        for i in range(0,len(Data)):
            token = Data['Token'][i].replace("\n","")
            token = token.replace(" ","")
            token = token.replace("\r","")
            segment = Data['Segment'][i].replace("\n","")
            segment = segment.replace(" ","")
            segment = segment.replace("\r","")
            if(segment == Data['Segment'][i]):
                segmentLen = Data['SegmentLen'][i]
            else:
                segmentLen = len(segment)
            if((token != "") & (token != " ") & (token != "nan")):
                writer.writerow([j,Data['Sentence#'][i],token,Data['Tag'][i],Data['Tag-Mixed'][i],segment,segmentLen])
                j += 1

# update the sentence number column
def cleanTheSentenceNum(outputName,inputName):
    Data = pd.read_csv(os.getcwd() + '/'+ inputName +'.csv')
    with open(outputName+'.csv', 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['#', 'Sentence#', 'Token', 'Tag', 'Tag-Mixed', 'Segment', 'SegmentLen'])
        j = 1
        sentenceNum = 1
        lastSentenceLen = 1
        for i in range(0,len(Data)):
            NSentence = Data['Sentence#'][i]
            if(lastSentenceLen != NSentence):
                sentenceNum += 1
            lastSentenceLen = NSentence
            writer.writerow([j,sentenceNum,Data['Token'][i],Data['Tag'][i],Data['Tag-Mixed'][i],Data['Segment'][i],Data['SegmentLen'][i]])
            j+=1

In [6]:
removespaceIntoken("TwitterData","TwitterData")
cleanTheSentenceNum("TwitterData","TwitterData")

In [2]:
# load the final data
sentenceToken= [ os.getcwd() + '/TwitterData.csv']
data = pd.read_csv(sentenceToken[0])
data

,#,Sentence#,Token,Tag,Tag-Mixed,Segment,SegmentLen
0,1,1,derasa,AR,AR,derasa,6
1,2,1,?,OTHER,OTHER,?,1
2,3,1,ya,AR,AR,ya,2
3,4,1,addy,NE.AR,NE.AR,addy,4
4,5,1,howa,AR,AR,howa,4
...,...,...,...,...,...,...,...
8584,8585,545,el,AR,AR,el,2
8585,8586,545,mafrood,AR,AR,mafrood,7
8586,8587,545,tekoon,AR,AR,tekoon,6
8587,8588,545,fein,AR,AR,fein,4


### Some data analysis

In [3]:
print(data.isnull().any())

#             False
Sentence#     False
Token         False
Tag           False
Tag-Mixed     False
Segment       False
SegmentLen    False
dtype: bool


In [4]:
print(data["Tag"].value_counts())
print(data["Tag-Mixed"].value_counts())

AR       5206
EN       2321
OTHER     748
MIXED     167
NE.AR      80
NE.EN      48
AMBIG      18
LANG3       1
Name: Tag, dtype: int64
AR             5206
EN             2321
OTHER           748
AR,EN           158
NE.AR            80
NE.EN            48
AMBIG            18
AR,OTHER,EN       3
EN,AR             3
AR,EN,AR          3
LANG3             1
Name: Tag-Mixed, dtype: int64


In [5]:
tokens = list(set(data["Token"].values))
len(tokens)

3937